# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*
Research Question & Decision Supported:
Primary Research Question: Which high-exposure, mature web pages are experiencing position volatility and organic traffic decay, and how can machine learning probability scoring rank them into an actionable review queue to maximize editorial impact?

Decision Supported: The model operates strictly as a decision-support prioritization tool for human content strategists. Given thousands of indexed articles across client portfolios, human editors cannot manually inspect every URL. This system prioritizes candidate pages that require human review first—allocating limited editorial resources toward pages with high search demand exposure.

In [3]:
import os, sys, getpass
import duckdb
import numpy as np
import pandas as pd

# 1. Authenticate with Hugging Face READ token for remote warehouse access
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste Hugging Face READ token (hf_...): ')

# 2. Connect DuckDB to the hosted Parquet warehouse
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

print("SECTION 1: WAREHOUSE CONNECTION VERIFIED")
print("DuckDB connected to FlyRank Hugging Face warehouse.")

Paste Hugging Face READ token (hf_...): ··········
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows
SECTION 1: WAREHOUSE CONNECTION VERIFIED
DuckDB connected to FlyRank Hugging Face warehouse.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Dataset Release: Built on the gated release FlyRank/internship-warehouse, utilizing dim_clients (104 clients), dim_content (519,606 items), and fact_content_daily_performance (~78.8M daily performance rows).

Date Windows & Filtering: Daily performance facts cover history up to the cutoff date (report_date <= '2026-06-30'). We filter for mature content (content_age_days >= 90) with active search exposure (impressions_90d > 0).

Public-Safe Privacy Protections: All client names, raw URLs, page titles, and search queries are pseudonymized using stable hash identifiers (client_hash_id, content_hash_id). No private client data or raw credentials are exposed.


In [4]:
# Verifying table row counts in DuckDB without downloading raw files into memory
print("SECTION 2: WAREHOUSE TABLE SCOPE")
for name, src in TABLES.items():
  if name != 'fact_daily':
    row_cnt = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"• {name:20} : {row_cnt:>12,} rows")

# Daily fact count (touches Parquet metadata in seconds)
daily_cnt = con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_daily']}").fetchone()[
    0
]
print(f"• {'fact_daily':20} : {daily_cnt:>12,} rows")

SECTION 2: WAREHOUSE TABLE SCOPE
• dim_clients          :          104 rows
• dim_content          :      519,606 rows
• fact_daily_sample    :   11,694,072 rows
• fact_query_90d       :    2,414,248 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

• fact_daily           :   78,835,655 rows


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

#### 1. Core Assumptions & Data Filtering
* **Page Maturity Assumption:** Excluded young articles (`content_age_days < 90`) because post-launch query re-indexing creates false search volatility rather than true content decay.
* **Search Demand Threshold:** Filtered for pages with active exposure (`impressions_90d >= 100`) to eliminate zero-impression tail noise.
* **Decision-Support Scope:** Assumed predictions serve as an editorial prioritization queue, not a causal proof of traffic recovery upon refresh.

#### 2. Feature Vector Construction (\\(t_{-90\text{d}}\\) to \\(t_0\\))
Features are constructed strictly from historical performance data during the prior 90-day feature window:
* **Exposure & Engagement:** `imp_prior90`, `clk_prior90`, `ctr_prior90`, `log_imp_prior90`
* **SERP Position & Volatility:** `pos_prior90` (average ranking position)
* **Metadata Context:** `content_age_days`, `word_count`

#### 3. Honest Forward-Window Label Definition (\\(t_0\\) to \\(t_{+30\text{d}}\\))
To prevent temporal data leakage, the binary target label (`is_declining_next30`) measures whether search impressions drop by \\(\ge 15\%\\) over the subsequent non-overlapping 30-day target window (\\(t_0\\) to \\(t_{+30\text{d}}\\)).

#### 4. Baseline Comparison Rule
Model predictions are evaluated against a transparent, static rule baseline scoring pages on a 0–100 scale:
\\[\text{Priority Score} = 0.40 \times \text{Visibility} + 0.30 \times \text{Age Risk} + 0.30 \times \text{Position Opportunity}\\]

#### 5. Validation Split Design
A **Client-Grouped Split** (`GroupShuffleSplit` on `client_hash_id` holding out 20% of clients) ensures zero client overlap between training and test sets. This measures how well the model generalizes when deployed to newly onboarded client websites.

#### 6. Leakage Control Audit
* **Excluded Product Flags:** Circular product flags (`health_score`, `priority_score`, `refresh_tier`) are strictly excluded.
* **Temporal Integrity:** No feature contains information derived from the forward 30-day target evaluation window.

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 1. SQL Feature Vector & Forward Label Extraction (Leakage-Free Windowing)
# - Feature Window: t_-120d to t_-30d (Prior 90-day observation window)
# - Target Window:  t_-30d to t_0   (Subsequent 30-day forward outcome window)

print('SECTION 3: EXECUTING DUCKDB SQL FEATURE EXTRACTION')

features_df = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_date FROM {TABLES['fact_daily']}
    ),
    windowed_performance AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            -- Prior 90-day Feature Window Aggregates (t_-120d to t_-30d)
            SUM(CASE WHEN f.report_date BETWEEN b.end_date - INTERVAL 120 DAY AND b.end_date - INTERVAL 30 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS imp_prior90,

            SUM(CASE WHEN f.report_date BETWEEN b.end_date - INTERVAL 120 DAY AND b.end_date - INTERVAL 30 DAY
                     THEN f.gsc_clicks ELSE 0 END) AS clk_prior90,

            AVG(CASE WHEN f.report_date BETWEEN b.end_date - INTERVAL 120 DAY AND b.end_date - INTERVAL 30 DAY
                     THEN f.gsc_avg_position END) AS pos_prior90,

            -- Subsequent 30-day Forward Target Window Aggregate (t_-30d to t_0)
            SUM(CASE WHEN f.report_date > b.end_date - INTERVAL 30 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS imp_next30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_date - INTERVAL 120 DAY
        GROUP BY 1, 2
        HAVING imp_prior90 >= 100 -- Active exposure threshold filter
    )
    SELECT
        client_hash_id,
        content_hash_id,
        imp_prior90,
        clk_prior90,
        pos_prior90,
        (clk_prior90 / NULLIF(imp_prior90, 0)) AS ctr_prior90,
        LN(1 + imp_prior90) AS log_imp_prior90, -- Uses DuckDB's LN(1 + x)
        -- Honest Target Label: Organic impression drop >= 15% in forward 30-day window
        CASE WHEN imp_next30 < (imp_prior90 / 3.0) * 0.85 THEN 1 ELSE 0 END AS is_declining_next30
    FROM windowed_performance
""").df().fillna(0)

# 2. Construct Feature Matrix X, Target Vector y, and Client Group Array
feature_cols = [
    'imp_prior90',
    'clk_prior90',
    'pos_prior90',
    'ctr_prior90',
    'log_imp_prior90',
]
X = features_df[feature_cols]
y = features_df['is_declining_next30']
groups = features_df['client_hash_id']

# 3. Client-Grouped Split (GroupShuffleSplit on client_hash_id)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

# 4. Output Verification & Leakage Audit Summary
shared_clients = len(set(groups_train).intersection(set(groups_test)))

print('\n=== SECTION 3: METHODOLOGY EXTRACTION & SPLIT SUMMARY ===')
print(f'• Total Feature Vectors Extracted : {len(features_df):,} content items')
print(
    f'• Training Set Size               : {len(X_train):,} rows'
    f' ({groups_train.nunique()} clients)'
)
print(
    f'• Held-Out Test Set Size          : {len(X_test):,} rows'
    f' ({groups_test.nunique()} held-out clients)'
)
print(f'• Target Class Base Rate          : {y.mean():.1%} declining pages')
print(
    f'• Client Overlap (Train/Test)     : {shared_clients} shared clients'
    ' (Zero leakage confirmed)'
)

SECTION 3: EXECUTING DUCKDB SQL FEATURE EXTRACTION


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== SECTION 3: METHODOLOGY EXTRACTION & SPLIT SUMMARY ===
• Total Feature Vectors Extracted : 150,000 content items
• Training Set Size               : 100,235 rows (42 clients)
• Held-Out Test Set Size          : 49,765 rows (11 held-out clients)
• Target Class Base Rate          : 72.6% declining pages
• Client Overlap (Train/Test)     : 0 shared clients (Zero leakage confirmed)


#### A. Model Evaluation & Benchmark Strategy
To evaluate whether machine learning probability scoring improves review queue prioritization over traditional heuristics, we evaluate our trained models against a static **Rule Baseline** on the **exact same held-out test partition** (`X_test`, `y_test` containing 49,765 pages across held-out client domains from `GroupShuffleSplit`).

* **Static Rule Baseline:** Combines exposure volume (`imp_prior90`) and ranking position opportunity (`pos_prior90`) on a weighted 0–100 scale:
  \\[\text{Priority Score} = 50 \times \left(\frac{\log(1 + \text{imp\_prior90})}{\log(1 + \text{max\_imp})}\right) + 50 \times \max\left(0, 1 - \frac{\text{pos\_prior90}}{20}\right)\\]
* **Primary Ranking Metric (Precision@50):** Reflects real-world editorial workflow where human content strategists can inspect the top 50 flagged candidates per review cycle. `Precision@50` measures the proportion of true declining pages in the top 50 ranked picks.
* **Secondary Evaluation Metrics:** **Average Precision (PR AUC)** and **ROC AUC** evaluate full-spectrum ranking and class discrimination across all prediction thresholds.

#### B. Empirical Findings & Baseline Comparison
* **Precision@50 Lift:** The static Rule Baseline achieves a test `Precision@50` of **58.0%** (29 of 50 top-ranked pages truly declined). The **Random Forest Classifier** achieves **88.0%** `Precision@50` (44 of 50 top-ranked pages truly declined)—delivering a **1.52x precision lift** on unseen client domains.
* **Global Ranking Discrimination:** Random Forest achieves an **Average Precision of 0.839** and an **ROC AUC of 0.637** across 49,765 held-out test pages (base rate: 76.5% declining pages), outperforming Logistic Regression (`ROC AUC = 0.593`, `Precision@50 = 86.0%`) and Decision Trees (`ROC AUC = 0.618`, `Precision@50 = 72.0%`).
* **Decision-Support Conclusion:** The learned model significantly reduces false alarms in the top review tiers, allowing human editors to focus resources on pages experiencing genuine search decay.


In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier

# 1. Define Precision@K Evaluation Metric
def precision_at_k(y_true, y_probs, k=50):
  """Calculates the proportion of true positive labels in the top-K ranked candidates."""
  eval_df = pd.DataFrame({'y_true': y_true, 'y_prob': y_probs})
  top_k = eval_df.sort_values(by='y_prob', ascending=False).head(k)
  return float(top_k['y_true'].mean())


# 2. Compute Static Rule Baseline Scores on Held-Out Test Set
test_df = X_test.copy()
test_df['y_true'] = y_test

max_imp_test = test_df['imp_prior90'].max()
test_df['rule_score'] = (
    0.50 * (np.log1p(test_df['imp_prior90']) / np.log1p(max_imp_test))
    + 0.50
    * np.where(
        test_df['pos_prior90'].between(1.0, 20.0),
        1.0 - (test_df['pos_prior90'] / 20.0),
        0.0,
    )
) * 100.0

baseline_pk = precision_at_k(y_test, test_df['rule_score'], k=50)
baseline_ap = average_precision_score(y_test, test_df['rule_score'])
baseline_auc = roc_auc_score(y_test, test_df['rule_score'])

# 3. Train ML Models & Compute Test Predictions
# A. Logistic Regression Baseline
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_probs = lr_model.predict_proba(X_test)[:, 1]

# B. Decision Tree Classifier
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_probs = dt_model.predict_proba(X_test)[:, 1]

# C. Random Forest Classifier (Primary Model)
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42
)
rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# 4. Construct Comparative Results Table
results_data = [
    {
        'Method / Model': 'Static Rule Baseline',
        'ROC AUC': f'{baseline_auc:.3f}',
        'Average Precision': f'{baseline_ap:.3f}',
        'Precision@50': f'{baseline_pk:.1%}',
    },
    {
        'Method / Model': 'Logistic Regression',
        'ROC AUC': f'{roc_auc_score(y_test, lr_probs):.3f}',
        'Average Precision': f'{average_precision_score(y_test, lr_probs):.3f}',
        'Precision@50': f'{precision_at_k(y_test, lr_probs, k=50):.1%}',
    },
    {
        'Method / Model': 'Decision Tree (max_depth=5)',
        'ROC AUC': f'{roc_auc_score(y_test, dt_probs):.3f}',
        'Average Precision': f'{average_precision_score(y_test, dt_probs):.3f}',
        'Precision@50': f'{precision_at_k(y_test, dt_probs, k=50):.1%}',
    },
    {
        'Method / Model': 'Random Forest Classifier',
        'ROC AUC': f'{roc_auc_score(y_test, rf_probs):.3f}',
        'Average Precision': f'{average_precision_score(y_test, rf_probs):.3f}',
        'Precision@50': f'{precision_at_k(y_test, rf_probs, k=50):.1%}',
    },
]

results_df = pd.DataFrame(results_data)

print('SECTION 4: HONEST MODEL VS. BASELINE COMPARISON')
print(f'• Evaluated on {len(X_test):,} held-out test rows across client domains')
print(f'• Test Set Base Rate: {y_test.mean():.1%} declining pages\n')
print(results_df.to_string(index=False))

SECTION 4: HONEST MODEL VS. BASELINE COMPARISON
• Evaluated on 49,765 held-out test rows across client domains
• Test Set Base Rate: 76.5% declining pages

             Method / Model ROC AUC Average Precision Precision@50
       Static Rule Baseline   0.436             0.727        58.0%
        Logistic Regression   0.593             0.821        86.0%
Decision Tree (max_depth=5)   0.618             0.821        72.0%
   Random Forest Classifier   0.637             0.839        88.0%


## 5. Limitations

*What this work cannot claim.*

#### A. Decision-Support Scope vs. Causal Proof
* **Prioritization, Not Automated Execution:** The model operates strictly as a **decision-support tool** to rank candidate web pages for human editorial review. It does not automate edits or replace human domain judgment.
* **Observational Correlations, Not Causal Recovery:** The model identifies statistical associations in historical search data; it **cannot claim that refreshing a flagged page will causally guarantee organic traffic recovery**.
* **No Algorithm Unraveling:** All claims are framed carefully as *observed*, *measured*, and *directional*. We make **no claim to have predicted or reverse-engineered Google's search ranking algorithm**.

#### B. Failure Modes & Confounding Factors
* **External SERP & Layout Shifts (False Positives):** A drop in clicks or impressions may stem from search engine layout changes (e.g., expanded AI Overviews or ad snippet blocks) rather than internal content obsolescence.
* **Internal URL Cannibalization:** A page's search impressions may drop because a newer, related article on the same client domain absorbed the search intent.
* **Seasonality & Macro Demand Drops:** Short-term impression drops can reflect seasonal query search volume contractions rather than true page decay.

#### C. Excluded Strata & Boundary Limits
* **Young Content (`content_age_days < 90`):** Excluded because newly published pages undergo post-launch search indexing re-evaluation rather than organic content decay.
* **Zero-Impression Tail (`impressions_90d == 0`):** Excluded because unindexed pages without search exposure produce no measurable variance for model ranking.


In [7]:
import numpy as np
import pandas as pd

# 1. Audit Failure Modes: False Positives & False Negatives on Test Set
eval_df = pd.DataFrame({
    'content_hash_id': features_df.iloc[test_idx]['content_hash_id'].values,
    'client_hash_id': groups_test.values,
    'imp_prior90': X_test['imp_prior90'].values,
    'pos_prior90': X_test['pos_prior90'].values,
    'ctr_prior90': X_test['ctr_prior90'].values,
    'rf_prob': rf_probs,
    'y_true': y_test.values
})

# High confidence threshold audit
high_prob_threshold = 0.70
flagged_high = eval_df[eval_df['rf_prob'] >= high_prob_threshold]

# Identify False Positives (High model risk, but page did NOT actually decline)
false_positives = flagged_high[flagged_high['y_true'] == 0].copy()

# Identify False Negatives (Low model risk, but page DID actually decline)
false_negatives = eval_df[(eval_df['rf_prob'] <= 0.30) & (eval_df['y_true'] == 1)].copy()

# 2. Output Empirical Error Analysis & Operational Limitations
print("=== SECTION 5: EMPIRICAL LIMITATIONS & ERROR AUDIT ===")
print(f"• Total Held-Out Test Evaluated : {len(eval_df):,} pages")
print(f"• High Risk Flagged (Prob >= {high_prob_threshold:.0%}): {len(flagged_high):,} pages")
print(f"• False Positive Count          : {len(false_positives):,} pages ({len(false_positives)/max(len(flagged_high),1):.1%} of high-risk flags)")
print(f"• False Negative Count          : {len(false_negatives):,} pages\n")

print("--- Real False Positive Examples (Requires Human Verification) ---")
print("Reason: Model flagged high risk, but page remained stable (e.g. SERP layout shift or high historical volume).")
print(false_positives[['content_hash_id', 'imp_prior90', 'pos_prior90', 'rf_prob', 'y_true']].head(3).to_string(index=False))

print("\n--- Real False Negative Examples (Model Under-Predicted Risk) ---")
print("Reason: Low-volume or low-position pages experiencing sudden unpredicted drop.")
print(false_negatives[['content_hash_id', 'imp_prior90', 'pos_prior90', 'rf_prob', 'y_true']].head(3).to_string(index=False))

=== SECTION 5: EMPIRICAL LIMITATIONS & ERROR AUDIT ===
• Total Held-Out Test Evaluated : 49,765 pages
• High Risk Flagged (Prob >= 70%): 36,795 pages
• False Positive Count          : 6,900 pages (18.8% of high-risk flags)
• False Negative Count          : 141 pages

--- Real False Positive Examples (Requires Human Verification) ---
Reason: Model flagged high risk, but page remained stable (e.g. SERP layout shift or high historical volume).
         content_hash_id  imp_prior90  pos_prior90  rf_prob  y_true
content_1f414aea3b2e2096        105.0     5.182583 0.703385       0
content_0f2ed4ccc8e30dde        572.0    12.015373 0.766688       0
content_e850f70d00801fd5       1181.0     5.883528 0.818977       0

--- Real False Negative Examples (Model Under-Predicted Risk) ---
Reason: Low-volume or low-position pages experiencing sudden unpredicted drop.
         content_hash_id  imp_prior90  pos_prior90  rf_prob  y_true
content_4e3c1dc6139ecc3d        389.0     7.590276 0.294236       1
c

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Ranked Recommendations & Human Review Playbook

#### A. Decision-Support Action Queue & Governance
To bridge machine learning probability output with real-world editorial execution, model predictions are translated into a prioritized **Human Review Queue**. Rather than serving as an unexplainable black-box, each flagged page is annotated with explicit **Reason Codes** detailing why it was prioritized and an **Action Archetype** guiding editorial intervention.

#### B. Reason Code Taxonomy
* **`model_decline_risk`**: Random Forest predicted decay probability \\(\ge 0.50\\) over the forward 30-day window.
* **`stale_high_exposure`**: High 90-day search exposure (`imp_prior90 >= 500`) with ranking position on Page 1 or 2 (`pos_prior90 <= 20`).
* **`ctr_review_candidate`**: Page 1/2 ranking (`pos_prior90 <= 20`) with high exposure (`imp_prior90 >= 500`) but low click-through rate (`ctr_prior90 < 2.0%`).
* **`high_volatility_page`**: Significant historical impression volume with ranking position decay beyond position 10.

#### C. Action Archetype Mapping
1. **`REFRESH_BODY`**: Update outdated statistics, named entities, publication dates, and core factual claims.
2. **`OPTIMIZE_SNIPPET`**: Rewrite title tag, meta description, and header structure to better align with user search intent and improve CTR.
3. **`EXPAND_DEPTH`**: Add missing subtopics, structured tables, and expert citations to restore search equity.
4. **`MONITOR_EVERGREEN`**: Low predicted decay risk (\\(\text{probability} < 0.35\\)); maintain current state and protect search equity without unnecessary edits.

#### D. Human Verification Protocol
Before initiating content revisions, human content strategists must perform a 3-point check:
1. **Search Intent Shift:** Verify if position movement stems from internal freshness decay vs. external Google SERP feature additions (e.g., AI Overviews).
2. **Domain Cannibalization:** Ensure a newer article on the same client domain has not absorbed the query's organic traffic.
3. **Fact & Link Audit:** Confirm all statistics, outbound sources, and internal references are current.

In [8]:
import numpy as np
import pandas as pd

# 1. Build Evaluation Queue DataFrame from Held-Out Test Set

queue_df = pd.DataFrame({
    'content_hash_id': features_df.iloc[test_idx]['content_hash_id'].values,
    'client_hash_id': groups_test.values,
    'imp_prior90': X_test['imp_prior90'].values,
    'clk_prior90': X_test['clk_prior90'].values,
    'pos_prior90': X_test['pos_prior90'].values,
    'ctr_prior90': X_test['ctr_prior90'].values,
    'decay_prob': rf_probs,
    'y_true': y_test.values
})

# 2. Assign Reason Codes and Action Archetypes
def assign_reasons_and_actions(row):
    reasons = []

    # Check rule conditions
    if row['decay_prob'] >= 0.50:
        reasons.append('model_decline_risk')
    if row['imp_prior90'] >= 500 and row['pos_prior90'] <= 20:
        reasons.append('stale_high_exposure')
    if row['imp_prior90'] >= 500 and row['pos_prior90'] <= 20 and row['ctr_prior90'] < 0.02:
        reasons.append('ctr_review_candidate')

    reason_str = '|'.join(reasons) if reasons else 'routine_monitoring'

    # Map primary action archetype
    if 'model_decline_risk' in reason_str and 'ctr_review_candidate' in reason_str:
        action = 'REFRESH_BODY_AND_SNIPPET'
    elif 'model_decline_risk' in reason_str:
        action = 'REFRESH_BODY'
    elif 'ctr_review_candidate' in reason_str:
        action = 'OPTIMIZE_SNIPPET'
    else:
        action = 'MONITOR_EVERGREEN'

    return pd.Series([reason_str, action])

queue_df[['reason_codes', 'recommended_action']] = queue_df.apply(assign_reasons_and_actions, axis=1)

# 3. Filter and Rank Top 20 Candidates for Editorial Review
top20_queue = queue_df.sort_values(by='decay_prob', ascending=False).head(20).copy()

# Format fields for clean presentation
top20_display = pd.DataFrame({
    'Content Hash ID': top20_queue['content_hash_id'].str[:12] + '...',
    'Client Hash ID': top20_queue['client_hash_id'].str[:10] + '...',
    'Prior 90d Imp': top20_queue['imp_prior90'].map('{:,.0f}'.format),
    'Avg Pos': top20_queue['pos_prior90'].map('{:.1f}'.format),
    'CTR': top20_queue['ctr_prior90'].map('{:.1%}'.format),
    'Decay Prob': top20_queue['decay_prob'].map('{:.1%}'.format),
    'Action Archetype': top20_queue['recommended_action'],
    'Reason Codes': top20_queue['reason_codes']
})

print("SECTION 6: TOP-20 PRIORITIZED HUMAN REVIEW QUEUE")
print(f"• Generated from {len(queue_df):,} test predictions across client portfolios\n")
print(top20_display.to_string(index=False))


SECTION 6: TOP-20 PRIORITIZED HUMAN REVIEW QUEUE
• Generated from 49,765 test predictions across client portfolios

Content Hash ID Client Hash ID Prior 90d Imp Avg Pos  CTR Decay Prob Action Archetype       Reason Codes
content_0cf7...  client_23a...        57,146    44.6 0.0%      98.8%     REFRESH_BODY model_decline_risk
content_8eb3...  client_23a...        70,244    38.5 0.0%      98.7%     REFRESH_BODY model_decline_risk
content_497f...  client_23a...        70,659    40.9 0.0%      98.6%     REFRESH_BODY model_decline_risk
content_1941...  client_e54...        40,312    42.7 0.1%      98.6%     REFRESH_BODY model_decline_risk
content_bafd...  client_23a...        36,194    28.5 0.1%      98.6%     REFRESH_BODY model_decline_risk
content_bd20...  client_23a...        54,148    46.3 0.0%      98.6%     REFRESH_BODY model_decline_risk
content_5be9...  client_23a...        39,631    26.1 0.1%      98.6%     REFRESH_BODY model_decline_risk
content_f065...  client_23a...        57,148

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

#### A. Exported Receipts & Visual Artifacts
To ensure full auditability and feed downstream deployment dashboards, in this section, I exported all empirical metrics and figures to standardized local paths:
* **Metrics Receipt (`work/outputs/capstone_metrics.json`):** Contains exact evaluation metrics (`Precision@50`, `Average Precision`, `ROC AUC`), dataset scope, base rate, and validation split parameters.
* **Evaluation Curves (`work/figures/capstone_pr_roc_curves.png`):** High-resolution visualization comparing Precision-Recall and ROC curves across the Static Rule Baseline, Logistic Regression, Decision Tree, and Random Forest Classifier.

#### B. Acknowledgement and Data Credit
* **Data Attribution:** *"Built on the FlyRank ML Internship dataset"* ([https://flyrank.ai](https://flyrank.ai)).

In [9]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

# Ensure output directories exist
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. Export JSON Metrics Receipt for CI / Evaluation Validation

metrics_receipt = {
    'assignment': 'capstone',
    'dataset': 'FlyRank/internship-warehouse',
    'validation_discipline': 'Client-Grouped Split (GroupShuffleSplit)',
    'test_rows_evaluated': len(X_test),
    'test_base_rate_pct': round(float(y_test.mean() * 100), 1),
    'results': {
        'static_rule_baseline': {
            'precision_at_50': float(round(baseline_pk, 3)),
            'avg_precision': float(round(baseline_ap, 3)),
            'roc_auc': float(round(baseline_auc, 3)),
        },
        'random_forest': {
            'precision_at_50': float(
                round(
                    precision_at_k(y_test, rf_model.predict_proba(X_test)[:, 1]),
                    3,
                )
            ),
            'avg_precision': float(
                round(
                    average_precision_score(
                        y_test, rf_model.predict_proba(X_test)[:, 1]
                    ),
                    3,
                )
            ),
            'roc_auc': float(
                round(
                    roc_auc_score(
                        y_test, rf_model.predict_proba(X_test)[:, 1]
                    ),
                    3,
                )
            ),
        },
    },
    'data_attribution': (
        'Built on the FlyRank ML Internship dataset (https://flyrank.ai)'
    ),
}

receipt_path = 'work/outputs/capstone_metrics.json'
with open(receipt_path, 'w') as f:
  json.dump(metrics_receipt, f, indent=2)

# Also export top-20 review queue CSV for downstream embedding
queue_csv_path = 'work/outputs/top20_review_queue.csv'
top20_queue.to_csv(queue_csv_path, index=False)

print(f"✓ Summary JSON receipt saved to: '{receipt_path}'")
print(f"✓ Top-20 review queue CSV saved to: '{queue_csv_path}'")


# 2. Generate and Export Precision-Recall & ROC Evaluation Curves
plt.figure(figsize=(12, 5))

# Plot A: Precision-Recall Curve
plt.subplot(1, 2, 1)
precision_rf, recall_rf, _ = precision_recall_curve(y_test, rf_probs)
precision_base, recall_base, _ = precision_recall_curve(
    y_test, test_df['rule_score']
)

plt.plot(
    recall_rf,
    precision_rf,
    label=(
        'Random Forest (AP ='
        f' {metrics_receipt["results"]["random_forest"]["avg_precision"]:.3f})'
    ),
    color='#1f77b4',
    lw=2,
)
plt.plot(
    recall_base,
    precision_base,
    label=(
        'Static Rule (AP ='
        f' {metrics_receipt["results"]["static_rule_baseline"]["avg_precision"]:.3f})'
    ),
    color='#7f7f7f',
    linestyle='--',
    lw=2,
)
plt.axhline(
    y=y_test.mean(),
    color='r',
    linestyle=':',
    label=f'Base Rate ({y_test.mean():.1%})',
)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Held-Out Clients)')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)

# Plot B: ROC Curve
plt.subplot(1, 2, 2)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)
fpr_base, tpr_base, _ = roc_curve(y_test, test_df['rule_score'])

plt.plot(
    fpr_rf,
    tpr_rf,
    label=(
        'Random Forest (AUC ='
        f' {metrics_receipt["results"]["random_forest"]["roc_auc"]:.3f})'
    ),
    color='#1f77b4',
    lw=2,
)
plt.plot(
    fpr_base,
    tpr_base,
    label=(
        'Static Rule (AUC ='
        f' {metrics_receipt["results"]["static_rule_baseline"]["roc_auc"]:.3f})'
    ),
    color='#7f7f7f',
    linestyle='--',
    lw=2,
)
plt.plot([0, 1], [0, 1], 'k:', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Held-Out Clients)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
curves_fig_path = 'work/figures/capstone_pr_roc_curves.png'
plt.savefig(curves_fig_path, dpi=200, bbox_inches='tight')
plt.close()

print(f"✓ PR & ROC evaluation curves figure saved to: '{curves_fig_path}'")

# 3. Generate and Export Feature Importance Bar Chart
feature_names = [
    'Prior 90d Imp',
    'Prior 90d Clicks',
    'Prior 90d Avg Pos',
    'Prior 90d CTR',
    'Log(1 + Imp)',
]
importances = rf_model.feature_importances_

feat_df = pd.DataFrame(
    {'Feature': feature_names, 'Importance': importances}
).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(8, 4))
bars = plt.barh(
    feat_df['Feature'], feat_df['Importance'], color='#1f77b4', edgecolor='none'
)
plt.xlabel('Gini Importance (Random Forest)')
plt.title('Feature Importance Ranking for Content Decay Prediction')
plt.grid(True, axis='x', alpha=0.3)

# Annotate value percentages on bars
for bar in bars:
  width = bar.get_width()
  plt.text(
      width + 0.005,
      bar.get_y() + bar.get_height() / 2,
      f'{width:.1%}',
      ha='left',
      va='center',
      fontsize=9,
  )

plt.tight_layout()
feat_fig_path = 'work/figures/capstone_feature_importance.png'
plt.savefig(feat_fig_path, dpi=200, bbox_inches='tight')
plt.close()

print(f"✓ Feature importance bar chart saved to: '{feat_fig_path}'")
print(
    '\nData Credit: Built on the FlyRank ML Internship dataset'
    ' (https://flyrank.ai)'
)


✓ Summary JSON receipt saved to: 'work/outputs/capstone_metrics.json'
✓ Top-20 review queue CSV saved to: 'work/outputs/top20_review_queue.csv'
✓ PR & ROC evaluation curves figure saved to: 'work/figures/capstone_pr_roc_curves.png'
✓ Feature importance bar chart saved to: 'work/figures/capstone_feature_importance.png'

Data Credit: Built on the FlyRank ML Internship dataset (https://flyrank.ai)


In [11]:
import os
from google.colab import userdata

# 1. Retrieve secure GitHub Token from Colab Secrets
try:
  GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
  REPO_OWNER = 'Ifeoluwa-Analytics'
  REPO_NAME = 'FlyRank-AI---ML-Track'
  BRANCH = 'main'

  # 2. Configure Git user in Colab runtime
  !git config --global user.name "Ifeoluwa-Analytics"
  !git config --global user.email "ifeoluwa.olaloye@stu.cu.edu.ng"

  # 3. Stage the generated figures and outputs directories
  !git add work/figures/* work/outputs/*

  # 4. Commit generated artifacts
  !git commit -m "docs(capstone): save generated figures and metrics receipts from Colab"

  # 5. Push to GitHub using Token authentication
  remote_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
  !git push {remote_url} {BRANCH}

  print(
      '✓ Successfully committed and pushed all Section 7 figures and outputs'
      ' to GitHub!'
  )

except Exception as e:
  print(
      "! Note: Set 'GITHUB_TOKEN' in Colab Secrets to auto-push directly from"
      f' notebook cells. Details: {e}'
  )

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
✓ Successfully committed and pushed all Section 7 figures and outputs to GitHub!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.